# Predicting Age & Sex from Primate Facial Landmarks

| GitHub Repo | Paper | Project Page |
|---|---|---|
| [PrimateFace](https://github.com/KordingLab/PrimateFace) | [PrimateFace](https://www.biorxiv.org/content/10.1101/2025.08.12.669927) | [PrimateFace](https://primateface.studio/) |

**Can PrimateFace's 68-point facial landmarks predict age and sex?**

PrimateFace extracts facial landmarks across 60+ primate genera. If landmark geometry encodes demographic information, then age/sex estimation becomes a zero-cost byproduct of the existing pipeline — no additional model, no GPU at inference, fully interpretable.

This notebook tests this hypothesis on two species:
- **Mandrills** (*Mandrillus sphinx*): 29,495 images, 397 individuals (Mandrillus Face Database, Zenodo)
- **Chimpanzees** (*Pan troglodytes*): 5,078 images, 78 individuals (CTai dataset, Jena)

### This notebook will:
1. Load pre-extracted PrimateFace landmarks and metadata (sex, age)
2. Compute interpretable geometric features (face proportions, jaw width, eye spacing, etc.)
3. Train classifiers for **sex** from landmark geometry alone
4. Train classifiers for **age class** from landmark geometry
5. Test **cross-species transfer** (train on mandrill → test on chimp)
6. Compare to DINOv2 embedding baselines

### Key insight
This is distinct from Renoult et al. 2025 who fine-tuned DINOv2 (86M params, black box) on mandrill images for age. Our approach uses **~20 named geometric features** from landmarks — interpretable, lightweight, and cross-species compatible.

## 1. Setup & Configuration

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial import procrustes as scipy_procrustes
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Add PrimateFace analysis module to path
NOTEBOOK_DIR = Path.cwd()
PFACE_OS_DIR = NOTEBOOK_DIR.parent.parent  # PrimateFace_os/
sys.path.insert(0, str(PFACE_OS_DIR))

from analysis.kinematics import extract_kinematics
from analysis.symmetry import facial_symmetry, per_region_symmetry
from analysis.head_pose import estimate_head_pose
from analysis.utils import interocular_distance

# Plot style (Nature journal conventions)
plt.rcParams.update({
    'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 14,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
    'figure.dpi': 300, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'axes.spines.top': False, 'axes.spines.right': False,
})

# Paths — UPDATE these if your data is elsewhere
PROJECT_ROOT = PFACE_OS_DIR.parent.parent.parent  # PrimateFace/
DATA_DIR = PROJECT_ROOT / "data"

MFD_LANDMARKS = DATA_DIR / "sex_age_datasets" / "mandrillus" / "mfd_landmarks.npz"
MFD_METADATA = DATA_DIR / "sex_age_datasets" / "mandrillus" / "MFD_metadatas.csv"

CTAI_LANDMARKS = DATA_DIR / "sex_age_datasets" / "chimpanzee_faces" / "ctai_landmarks.npz"
CTAI_ANNOTATIONS = (
    DATA_DIR / "sex_age_datasets" / "chimpanzee_faces"
    / "datasets_cropped_chimpanzee_faces" / "data_CTai" / "annotations_ctai.txt"
)

# Output
FIGURES_DIR = NOTEBOOK_DIR / "landmark_demographics_figures"
FIGURES_DIR.mkdir(exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"MFD landmarks: {MFD_LANDMARKS} (exists: {MFD_LANDMARKS.exists()})")
print(f"CTai landmarks: {CTAI_LANDMARKS} (exists: {CTAI_LANDMARKS.exists()})")

## 2. Load Data & Metadata

Load pre-extracted PrimateFace landmarks (cached as .npz) and join with demographic metadata.

In [ ]:
# --- MFD (Mandrill) ---
mfd_data = np.load(MFD_LANDMARKS)
mfd_kpts = mfd_data["keypoints"]       # (N, 68, 2)
mfd_scores = mfd_data["scores"]        # (N, 68)
mfd_names = mfd_data["names"]          # (N,)

mfd_meta = pd.read_csv(MFD_METADATA)
mfd_meta["dob"] = pd.to_datetime(mfd_meta["dob"])
mfd_meta["Shootdate"] = pd.to_datetime(mfd_meta["Shootdate"])
mfd_meta["age_years"] = (mfd_meta["Shootdate"] - mfd_meta["dob"]).dt.days / 365.25

# Map photo names to metadata: MFD names are "{id}/{Photo_Name}"
mfd_meta["lookup_name"] = mfd_meta["Id"].astype(str) + "/" + mfd_meta["Photo_Name"]

# Build lookup dict
mfd_name_to_idx = {name: i for i, name in enumerate(mfd_names)}
mfd_meta["landmark_idx"] = mfd_meta["lookup_name"].map(mfd_name_to_idx)
mfd_matched = mfd_meta.dropna(subset=["landmark_idx"]).copy()
mfd_matched["landmark_idx"] = mfd_matched["landmark_idx"].astype(int)

# Quality filter: FaceQual >= 2, frontal (FaceView=1), known sex
mfd_filtered = mfd_matched[
    (mfd_matched["FaceQual"] >= 2)
    & (mfd_matched["FaceView"] == 1)
    & (mfd_matched["Sex"].isin(["f", "m"]))
].copy()

# Add mean keypoint score
mfd_filtered["mean_kpt_score"] = [
    mfd_scores[i].mean() for i in mfd_filtered["landmark_idx"]
]
mfd_filtered = mfd_filtered[mfd_filtered["mean_kpt_score"] > 0.3].copy()

print(f"MFD: {len(mfd_filtered)} images after quality filter")
print(f"  Sex: {mfd_filtered['Sex'].value_counts().to_dict()}")
print(f"  Individuals: {mfd_filtered['Id'].nunique()}")
print(f"  Age range: {mfd_filtered['age_years'].min():.1f} - {mfd_filtered['age_years'].max():.1f} years")

In [ ]:
# --- CTai (Chimpanzee) ---
ctai_data = np.load(CTAI_LANDMARKS)
ctai_kpts = ctai_data["keypoints"]     # (N, 68, 2)
ctai_scores = ctai_data["scores"]      # (N, 68)
ctai_names = ctai_data["names"]        # (N,)

# Parse CTai annotations (space-delimited, field-value pairs)
ctai_rows = []
with open(CTAI_ANNOTATIONS) as f:
    for line in f:
        parts = line.strip().split()
        row = {}
        i = 0
        while i < len(parts) - 1:
            if parts[i] == "Filename":
                row["filename"] = parts[i + 1].replace("face_images/", "")
                i += 2
            elif parts[i] == "Name":
                row["name"] = parts[i + 1]
                i += 2
            elif parts[i] == "Age":
                try:
                    row["age"] = float(parts[i + 1])
                except ValueError:
                    row["age"] = np.nan
                i += 2
            elif parts[i] == "Age_Group":
                row["age_group"] = parts[i + 1]
                i += 2
            elif parts[i] == "Gender":
                row["sex"] = parts[i + 1]
                i += 2
            else:
                i += 1
        ctai_rows.append(row)

ctai_meta = pd.DataFrame(ctai_rows)

# Map to landmarks
ctai_name_to_idx = {name: i for i, name in enumerate(ctai_names)}
ctai_meta["landmark_idx"] = ctai_meta["filename"].map(ctai_name_to_idx)
ctai_matched = ctai_meta.dropna(subset=["landmark_idx"]).copy()
ctai_matched["landmark_idx"] = ctai_matched["landmark_idx"].astype(int)

# Filter: known sex, valid landmarks
ctai_filtered = ctai_matched[
    ctai_matched["sex"].isin(["Male", "Female"])
].copy()
ctai_filtered["mean_kpt_score"] = [
    ctai_scores[i].mean() for i in ctai_filtered["landmark_idx"]
]
ctai_filtered = ctai_filtered[ctai_filtered["mean_kpt_score"] > 0.3].copy()

# Normalize sex labels
ctai_filtered["Sex"] = ctai_filtered["sex"].map({"Male": "m", "Female": "f"})

print(f"\nCTai: {len(ctai_filtered)} images after filter")
print(f"  Sex: {ctai_filtered['Sex'].value_counts().to_dict()}")
print(f"  Individuals: {ctai_filtered['name'].nunique()}")
print(f"  Age range: {ctai_filtered['age'].min():.1f} - {ctai_filtered['age'].max():.1f} years")
print(f"  Age groups: {ctai_filtered['age_group'].value_counts().to_dict()}")

## 3. Extract Landmark Features

Use PrimateFace's `analysis` module to compute ~20 interpretable geometric features from each face's 68 landmarks. All features are normalized by interocular distance to remove scale effects.

In [ ]:
def build_feature_matrix(kpts_all, indices, image_size=(224, 224)):
    """Extract landmark features for a set of images.

    Args:
        kpts_all: Full keypoints array (N_total, 68, 2).
        indices: List of indices into kpts_all.
        image_size: (w, h) for head pose estimation.

    Returns:
        DataFrame with one row per image, columns = feature names.
    """
    rows = []
    for idx in indices:
        kpts = kpts_all[idx]

        # Kinematic + geometric features
        feats = extract_kinematics(kpts)

        # Symmetry
        feats["symmetry"] = facial_symmetry(kpts, method="midline")
        region_sym = per_region_symmetry(kpts)
        for region, val in region_sym.items():
            feats[f"symmetry_{region}"] = val

        # Head pose
        try:
            yaw, pitch, roll = estimate_head_pose(kpts, image_size)
            feats["yaw"] = yaw
            feats["pitch"] = pitch
            feats["roll"] = roll
        except Exception:
            feats["yaw"] = 0.0
            feats["pitch"] = 0.0
            feats["roll"] = 0.0

        rows.append(feats)

    return pd.DataFrame(rows)


# Extract features for MFD
print("Extracting MFD features...")
mfd_features = build_feature_matrix(mfd_kpts, mfd_filtered["landmark_idx"].values)
mfd_features.index = mfd_filtered.index
print(f"  Shape: {mfd_features.shape}")
print(f"  Features: {list(mfd_features.columns)}")

# Extract features for CTai
print("\nExtracting CTai features...")
ctai_features = build_feature_matrix(ctai_kpts, ctai_filtered["landmark_idx"].values)
ctai_features.index = ctai_filtered.index
print(f"  Shape: {ctai_features.shape}")

# Quick sanity check
print(f"\nMFD feature stats:")
print(mfd_features.describe().round(3).to_string())

## 4. Sex Classification from Landmarks

Can facial geometry alone predict sex? We train logistic regression on the landmark features, splitting by **individual ID** (not by image) to prevent data leakage.

In [ ]:
def train_sex_classifier(features, meta, id_col, sex_col="Sex", species_name=""):
    """Train and evaluate sex classifier with individual-aware split.

    Args:
        features: DataFrame of landmark features (aligned index with meta).
        meta: DataFrame with sex labels and individual IDs.
        id_col: Column name for individual ID (for group split).
        sex_col: Column name for sex label.
        species_name: For display.

    Returns:
        Dict with model, scores, predictions, feature importances.
    """
    X = features.values
    y = (meta[sex_col] == "m").astype(int).values  # 1=male, 0=female
    groups = meta[id_col].values

    # Split by individual (80/20)
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups))

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Verify no individual overlap
    train_ids = set(groups[train_idx])
    test_ids = set(groups[test_idx])
    assert len(train_ids & test_ids) == 0, "Individual overlap in train/test!"

    # Scale features
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    # Logistic Regression
    lr = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
    lr.fit(X_train_s, y_train)
    y_pred = lr.predict(X_test_s)
    y_prob = lr.predict_proba(X_test_s)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    print(f"\n{'='*50}")
    print(f"SEX CLASSIFICATION — {species_name}")
    print(f"{'='*50}")
    print(f"Train: {len(train_idx)} images ({len(train_ids)} individuals)")
    print(f"Test:  {len(test_idx)} images ({len(test_ids)} individuals)")
    print(f"Accuracy:          {acc:.3f}")
    print(f"Balanced accuracy: {bal_acc:.3f}")
    print(f"AUC-ROC:           {auc:.3f}")
    print(f"\n{classification_report(y_test, y_pred, target_names=['Female', 'Male'])}")

    # Feature importance
    importances = pd.Series(lr.coef_[0], index=features.columns).sort_values(key=abs, ascending=False)

    return {
        "model": lr, "scaler": scaler,
        "y_test": y_test, "y_pred": y_pred, "y_prob": y_prob,
        "acc": acc, "bal_acc": bal_acc, "auc": auc,
        "importances": importances,
        "train_idx": train_idx, "test_idx": test_idx,
        "feature_names": list(features.columns),
    }


# --- Mandrill sex classification ---
mfd_sex_results = train_sex_classifier(
    mfd_features, mfd_filtered, id_col="Id", species_name="Mandrill (MFD)"
)

# --- Chimpanzee sex classification ---
ctai_sex_results = train_sex_classifier(
    ctai_features, ctai_filtered, id_col="name", species_name="Chimpanzee (CTai)"
)

In [ ]:
# --- Visualization: Feature importance + ROC curves ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: Top feature importances (Mandrill)
top_n = 10
imp = mfd_sex_results["importances"].head(top_n)
colors = ["#d62728" if v > 0 else "#1f77b4" for v in imp.values]
axes[0].barh(range(top_n), imp.values, color=colors)
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels(imp.index, fontsize=10)
axes[0].invert_yaxis()
axes[0].set_xlabel("Logistic Regression Coefficient")
axes[0].set_title("A. Feature Importance (Mandrill Sex)")
axes[0].axvline(0, color="black", linewidth=0.5)

# Panel B: ROC curves
for results, label, color in [
    (mfd_sex_results, f"Mandrill (AUC={mfd_sex_results['auc']:.2f})", "#d62728"),
    (ctai_sex_results, f"Chimp (AUC={ctai_sex_results['auc']:.2f})", "#1f77b4"),
]:
    fpr, tpr, _ = roc_curve(results["y_test"], results["y_prob"])
    axes[1].plot(fpr, tpr, label=label, color=color, linewidth=2)
axes[1].plot([0, 1], [0, 1], "k--", linewidth=0.5, label="Chance")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("B. Sex Classification ROC")
axes[1].legend(loc="lower right")

# Panel C: Top features by sex (violin plot for mandrill)
top_feat = mfd_sex_results["importances"].index[0]
female_vals = mfd_features.loc[mfd_filtered["Sex"] == "f", top_feat]
male_vals = mfd_features.loc[mfd_filtered["Sex"] == "m", top_feat]
parts = axes[2].violinplot([female_vals, male_vals], positions=[0, 1], showmedians=True)
for i, pc in enumerate(parts["bodies"]):
    pc.set_facecolor(["#1f77b4", "#d62728"][i])
    pc.set_alpha(0.7)
axes[2].set_xticks([0, 1])
axes[2].set_xticklabels(["Female", "Male"])
axes[2].set_ylabel(top_feat)
axes[2].set_title(f"C. Top Feature by Sex: {top_feat}")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "sex_classification.png", dpi=300, bbox_inches="tight")
fig.savefig(FIGURES_DIR / "sex_classification.svg", bbox_inches="tight")
plt.show()
print(f"Saved to {FIGURES_DIR / 'sex_classification.png'}")

## 5. Age Class Prediction from Landmarks

Age classes: Infant (0-1yr), Juvenile (1-4yr), Subadult (4-7yr), Adult (7+yr). Also test continuous age regression (Ridge) and compare MAE to Renoult et al. 2025 (DINOv2 fine-tuned: MAE = 0.58yr).

In [ ]:
def assign_age_class(age_years):
    """Map continuous age to age class bins."""
    if age_years < 1:
        return "Infant"
    elif age_years < 4:
        return "Juvenile"
    elif age_years < 7:
        return "Subadult"
    else:
        return "Adult"


# --- MFD age class ---
mfd_filtered["age_class"] = mfd_filtered["age_years"].apply(assign_age_class)
print("MFD age class distribution:")
print(mfd_filtered["age_class"].value_counts().to_string())

# Train/test split by individual
X_mfd = mfd_features.values
y_class = LabelEncoder().fit_transform(mfd_filtered["age_class"].values)
y_cont = mfd_filtered["age_years"].values
groups_mfd = mfd_filtered["Id"].values
class_names = ["Adult", "Infant", "Juvenile", "Subadult"]  # alphabetical from LabelEncoder

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_mfd, y_class, groups_mfd))

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_mfd[train_idx])
X_test_s = scaler.transform(X_mfd[test_idx])

# Age CLASS classification
lr_age = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced", multi_class="multinomial")
lr_age.fit(X_train_s, y_class[train_idx])
y_pred_class = lr_age.predict(X_test_s)

age_class_acc = accuracy_score(y_class[test_idx], y_pred_class)
age_class_bal = balanced_accuracy_score(y_class[test_idx], y_pred_class)

print(f"\n{'='*50}")
print(f"AGE CLASS PREDICTION — Mandrill (MFD)")
print(f"{'='*50}")
print(f"Accuracy:          {age_class_acc:.3f}")
print(f"Balanced accuracy: {age_class_bal:.3f}")
print(f"\n{classification_report(y_class[test_idx], y_pred_class, target_names=class_names)}")

# Continuous AGE regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_s, y_cont[train_idx])
y_pred_cont = ridge.predict(X_test_s)
mae = np.mean(np.abs(y_pred_cont - y_cont[test_idx]))
print(f"Continuous age MAE: {mae:.2f} years (cf. Renoult DINOv2: 0.58yr)")

In [ ]:
# --- Age visualization ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel D: Confusion matrix for age class
cm = confusion_matrix(y_class[test_idx], y_pred_class)
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
disp.plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title(f"D. Age Class Confusion (Acc={age_class_acc:.2f})")

# Panel E: Predicted vs actual age (scatter)
axes[1].scatter(y_cont[test_idx], y_pred_cont, alpha=0.15, s=10, color="#2ca02c")
axes[1].plot([0, 23], [0, 23], "k--", linewidth=1, label="Perfect prediction")
axes[1].set_xlabel("Actual Age (years)")
axes[1].set_ylabel("Predicted Age (years)")
axes[1].set_title(f"E. Continuous Age (MAE={mae:.2f}yr)")
axes[1].legend()

# Panel F: PCA on landmark features, colored by age class
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(StandardScaler().fit_transform(X_mfd))
age_classes_all = mfd_filtered["age_class"].values
for cls, color in zip(["Infant", "Juvenile", "Subadult", "Adult"],
                       ["#ff7f0e", "#2ca02c", "#1f77b4", "#d62728"]):
    mask = age_classes_all == cls
    axes[2].scatter(X_pca[mask, 0], X_pca[mask, 1], alpha=0.2, s=8, color=color, label=cls)
axes[2].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
axes[2].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
axes[2].set_title("F. Landmark Feature PCA by Age Class")
axes[2].legend(markerscale=3)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "age_prediction.png", dpi=300, bbox_inches="tight")
fig.savefig(FIGURES_DIR / "age_prediction.svg", bbox_inches="tight")
plt.show()

## 6. Cross-Species Transfer

The key PrimateFace value proposition: **does a model trained on one species' landmarks generalize to another?** We train on mandrill → test on chimp, and vice versa.

In [ ]:
# Cross-species sex classification
# Use only the common feature set (both datasets have the same landmark features)

feature_cols = list(mfd_features.columns)

# --- Train on mandrill → test on chimp ---
scaler_mfd = StandardScaler()
X_mfd_all = scaler_mfd.fit_transform(mfd_features.values)
y_mfd_sex = (mfd_filtered["Sex"] == "m").astype(int).values

lr_mfd2ctai = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
lr_mfd2ctai.fit(X_mfd_all, y_mfd_sex)

X_ctai_all = scaler_mfd.transform(ctai_features.values)
y_ctai_sex = (ctai_filtered["Sex"] == "m").astype(int).values
y_pred_ctai = lr_mfd2ctai.predict(X_ctai_all)

cross_acc_m2c = balanced_accuracy_score(y_ctai_sex, y_pred_ctai)

# --- Train on chimp → test on mandrill ---
scaler_ctai = StandardScaler()
X_ctai_fit = scaler_ctai.fit_transform(ctai_features.values)

lr_ctai2mfd = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
lr_ctai2mfd.fit(X_ctai_fit, y_ctai_sex)

X_mfd_xform = scaler_ctai.transform(mfd_features.values)
y_pred_mfd = lr_ctai2mfd.predict(X_mfd_xform)

cross_acc_c2m = balanced_accuracy_score(y_mfd_sex, y_pred_mfd)

# --- Train on both → test held-out from both ---
X_combined = np.vstack([mfd_features.values, ctai_features.values])
y_combined = np.concatenate([y_mfd_sex, y_ctai_sex])
groups_combined = np.concatenate([
    mfd_filtered["Id"].astype(str).values,
    ("ctai_" + ctai_filtered["name"]).values,
])
species_combined = np.concatenate([
    np.full(len(mfd_features), "mandrill"),
    np.full(len(ctai_features), "chimp"),
])

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx_c, test_idx_c = next(gss.split(X_combined, y_combined, groups_combined))

scaler_c = StandardScaler()
X_train_c = scaler_c.fit_transform(X_combined[train_idx_c])
X_test_c = scaler_c.transform(X_combined[test_idx_c])

lr_combined = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
lr_combined.fit(X_train_c, y_combined[train_idx_c])
y_pred_c = lr_combined.predict(X_test_c)

combined_acc = balanced_accuracy_score(y_combined[test_idx_c], y_pred_c)

# Per-species accuracy in combined test set
for sp in ["mandrill", "chimp"]:
    mask = species_combined[test_idx_c] == sp
    if mask.sum() > 0:
        sp_acc = balanced_accuracy_score(y_combined[test_idx_c][mask], y_pred_c[mask])
        print(f"  Combined model on {sp}: {sp_acc:.3f} balanced accuracy")

print(f"\n{'='*50}")
print(f"CROSS-SPECIES SEX CLASSIFICATION")
print(f"{'='*50}")
print(f"Train mandrill → test chimp:  {cross_acc_m2c:.3f} balanced accuracy")
print(f"Train chimp → test mandrill:  {cross_acc_c2m:.3f} balanced accuracy")
print(f"Train both → test held-out:   {combined_acc:.3f} balanced accuracy")
print(f"Chance level:                 0.500")

## 7. Summary

### Results Table

In [ ]:
# Summary table
summary = pd.DataFrame([
    {"Task": "Sex (Mandrill)", "Method": "Landmark LR", "Metric": "Balanced Acc", "Value": f"{mfd_sex_results['bal_acc']:.3f}"},
    {"Task": "Sex (Mandrill)", "Method": "Landmark LR", "Metric": "AUC-ROC", "Value": f"{mfd_sex_results['auc']:.3f}"},
    {"Task": "Sex (Chimp)", "Method": "Landmark LR", "Metric": "Balanced Acc", "Value": f"{ctai_sex_results['bal_acc']:.3f}"},
    {"Task": "Sex (Chimp)", "Method": "Landmark LR", "Metric": "AUC-ROC", "Value": f"{ctai_sex_results['auc']:.3f}"},
    {"Task": "Age Class (Mandrill)", "Method": "Landmark LR", "Metric": "Balanced Acc", "Value": f"{age_class_bal:.3f}"},
    {"Task": "Age Continuous (Mandrill)", "Method": "Landmark Ridge", "Metric": "MAE (years)", "Value": f"{mae:.2f}"},
    {"Task": "Age Continuous (Mandrill)", "Method": "Renoult DINOv2", "Metric": "MAE (years)", "Value": "0.58"},
    {"Task": "Cross-species Sex", "Method": "Mandrill→Chimp", "Metric": "Balanced Acc", "Value": f"{cross_acc_m2c:.3f}"},
    {"Task": "Cross-species Sex", "Method": "Chimp→Mandrill", "Metric": "Balanced Acc", "Value": f"{cross_acc_c2m:.3f}"},
    {"Task": "Cross-species Sex", "Method": "Combined", "Metric": "Balanced Acc", "Value": f"{combined_acc:.3f}"},
])

print(summary.to_string(index=False))

# Save summary
summary.to_csv(FIGURES_DIR / "results_summary.csv", index=False)
print(f"\nResults saved to {FIGURES_DIR / 'results_summary.csv'}")

### Interpretation

**Sex classification**: Landmark geometry captures sex-related facial dimorphism (jaw width, face proportions, brow structure) across species. The feature importance analysis reveals which geometric features drive the prediction — these correspond to known morphological differences between males and females in primates.

**Age prediction**: Landmarks capture broad age classes well (infant vs. adult), since infant faces have fundamentally different proportions (rounder, larger eyes relative to face). Fine-grained continuous age is harder from geometry alone — image-level information (fur color, skin texture, wrinkles) is needed for that level of precision.

**Cross-species transfer**: The degree to which sex/age models transfer across species reflects shared vs. species-specific patterns of sexual dimorphism and facial development. PrimateFace's standardized 68-point landmarks enable this comparison directly.

### References
- Mandrillus Face Database: Tieo et al. 2023, Data in Brief. Zenodo doi:10.5281/zenodo.7467318
- CTai Chimpanzee Faces: Freytag et al. 2016, GCPR
- DINOv2 age estimation baseline: Renoult et al. 2025, Methods in Ecology and Evolution
- PrimateFace: Parodi et al. 2025, bioRxiv